# Лабораторная работа 8

## Сбор данных в интернете


In [1]:
import json
import re
import ssl
from collections import Counter
import certifi
from html import unescape
from pathlib import Path
from urllib.parse import urlencode, urljoin
from urllib.request import Request, urlopen


NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
STATIC_MAP_URL = "https://static-maps.yandex.ru/1.x/"
YANDEX_GEOCODER_URL = "https://geocode-maps.yandex.ru/v1/"
YANDEX_GEOCODER_API_KEY = "0aa0ce3c-c817-4d77-9cff-38f4b5a1bb38"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Lab8Notebook/1.0)"
}
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())


def build_url(base_url, params=None):
    if not params:
        return base_url
    return f"{base_url}?{urlencode(params)}"


def fetch_text(url, params=None):
    request = Request(build_url(url, params), headers=HEADERS)
    with urlopen(request, timeout=30, context=SSL_CONTEXT) as response:
        return response.read().decode("utf-8")


def fetch_json(url, params=None):
    return json.loads(fetch_text(url, params))


def download_file(url, filename, params=None):
    request = Request(build_url(url, params), headers=HEADERS)
    with urlopen(request, timeout=30, context=SSL_CONTEXT) as response:
        data = response.read()
    path = Path(filename)
    path.write_bytes(data)
    return path.resolve()


def nominatim_search(query, limit=1):
    data = fetch_json(
        NOMINATIM_URL,
        {
            "q": query,
            "format": "jsonv2",
            "limit": limit,
            "addressdetails": 1,
            "accept-language": "ru",
        },
    )
    if not data:
        raise ValueError(f"Не удалось найти объект по запросу: {query}")
    return data


def record_coords(record):
    return float(record["lat"]), float(record["lon"])

def address_part(record, *keys):
    address = record.get("address", {})
    for key in keys:
        value = address.get(key)
        if value:
            return value
    return None


def federal_district(record):
    return address_part(record, "region")


def region_name(record):
    return address_part(record, "state", "region", "county")


def print_json(record):
    print(json.dumps(record, ensure_ascii=False, indent=2))


def yandex_geocode(query):
    response = fetch_json(
        YANDEX_GEOCODER_URL,
        {
            "apikey": YANDEX_GEOCODER_API_KEY,
            "geocode": query,
            "format": "json",
            "lang": "ru_RU",
        },
    )
    if "response" not in response:
        raise ValueError(response.get("message", "Ошибка Яндекс.Геокодера"))
    return response


def yandex_geoobject(response):
    members = response["response"]["GeoObjectCollection"]["featureMember"]
    if not members:
        raise ValueError("Яндекс.Геокодер не вернул результатов")
    return members[0]["GeoObject"]


def yandex_coords(geoobject):
    lon, lat = map(float, geoobject["Point"]["pos"].split())
    return lat, lon


def yandex_components(geoobject):
    return geoobject["metaDataProperty"]["GeocoderMetaData"]["Address"].get("Components", [])


def yandex_component_name(geoobject, kind):
    for component in yandex_components(geoobject):
        if component.get("kind") == kind:
            return component.get("name")
    return None


def yandex_postcode(geoobject):
    return geoobject["metaDataProperty"]["GeocoderMetaData"]["Address"].get("postal_code")


def yandex_federal_district(geoobject):
    for component in yandex_components(geoobject):
        name = component.get("name", "")
        if "федеральный округ" in name.lower():
            return name
    return None


## Работа с API через браузер

### №1
Yandex.Maps Static API позволяет получить изображение нужного фрагмента карты, которое можно разместить на сайте или в приложении. Такое изображение оптимизировано, «весит» не очень много и загружается быстро даже при медленном Интернете.

Static API возвращает изображение карты в ответ на HTTPS-запрос. Добавляя в URL разные параметры и задавая их значения, вы можете определить центр карты, ее размер и область показа, отметить нужные объекты и даже отобразить пробки. При этом при каждом новом запросе будет возвращаться изображение с актуальными данными.

Документация для Static API находится на странице:
`https://tech.yandex.ru/maps/doc/staticapi/1.x/dg/concepts/input_params-docpage/`

Посмотрите, что означают параметры `ll`, `spn`, `l` и с помощью запросов к API через браузер получите:

a) Крупномасштабную схему с КемГУ
b) Крупномасштабную схему района, в котором вы живете
c) Крупномасштабную схему города, в котором вы родились
d) Спутниковый снимок Эйфелевой башни
e) Спутниковый снимок Авачинского вулкана
f) Спутниковый снимок озера Байкал
g) Спутниковый снимок космодрома Байконур


In [2]:
static_api_examples = {
    "КемГУ": {"ll": "86.0909082,55.3518137", "spn": "0.01,0.006", "l": "map"},
    "Ленинский район Кемерово": {"ll": "86.162,55.345", "spn": "0.09,0.05", "l": "map"},
    "Кемерово": {"ll": "86.0872,55.3547", "spn": "0.32,0.18", "l": "map"},
    "Эйфелева башня": {"ll": "2.2945,48.8584", "spn": "0.01,0.01", "l": "sat"},
    "Авачинский вулкан": {"ll": "158.8360,53.2570", "spn": "0.08,0.05", "l": "sat"},
    "Озеро Байкал": {"ll": "108.0,53.5", "spn": "6.0,3.5", "l": "sat"},
    "Космодром Байконур": {"ll": "63.3070,45.9640", "spn": "0.7,0.45", "l": "sat"},
}

for title, params in static_api_examples.items():
    print(title)
    print(build_url(STATIC_MAP_URL, params))
    print()


КемГУ
https://static-maps.yandex.ru/1.x/?ll=86.0909082%2C55.3518137&spn=0.01%2C0.006&l=map

Ленинский район Кемерово
https://static-maps.yandex.ru/1.x/?ll=86.162%2C55.345&spn=0.09%2C0.05&l=map

Кемерово
https://static-maps.yandex.ru/1.x/?ll=86.0872%2C55.3547&spn=0.32%2C0.18&l=map

Эйфелева башня
https://static-maps.yandex.ru/1.x/?ll=2.2945%2C48.8584&spn=0.01%2C0.01&l=sat

Авачинский вулкан
https://static-maps.yandex.ru/1.x/?ll=158.8360%2C53.2570&spn=0.08%2C0.05&l=sat

Озеро Байкал
https://static-maps.yandex.ru/1.x/?ll=108.0%2C53.5&spn=6.0%2C3.5&l=sat

Космодром Байконур
https://static-maps.yandex.ru/1.x/?ll=63.3070%2C45.9640&spn=0.7%2C0.45&l=sat



### №2
Геокодер помогает определить координаты объекта по его адресу или, наоборот, установить адрес по координатам. К геокодеру можно также обращаться по протоколу HTTPS.

Ознакомьтесь с документацией, попробуйте сделать запросы, чтобы понять, что означают ответы геокодера, и ответьте на следующие вопросы (для каждого пункта укажите запрос, который использовали, и полученный ответ):

a) Получите координаты Якутска и Магадана в формате JSON. Какой город находится севернее: Якутск или Магадан?
b) Получите координаты вашего родного города и города Торонто в формате JSON. Какой город из них находится южнее?
c) Определите к каким федеральным округам относятся города: Хабаровск, Уфа, Нижний Новгород, Калининград, ваш родной город.
d) Узнайте почтовый индекс КемГУ.


In [3]:
browser_home_city = "Кемерово"

yakutsk_query = "Якутск"
magadan_query = "Магадан"
yakutsk_url = build_url(YANDEX_GEOCODER_URL, {"apikey": YANDEX_GEOCODER_API_KEY, "geocode": yakutsk_query, "format": "json", "lang": "ru_RU"})
magadan_url = build_url(YANDEX_GEOCODER_URL, {"apikey": YANDEX_GEOCODER_API_KEY, "geocode": magadan_query, "format": "json", "lang": "ru_RU"})
yakutsk_response = yandex_geocode(yakutsk_query)
magadan_response = yandex_geocode(magadan_query)
yakutsk = yandex_geoobject(yakutsk_response)
magadan = yandex_geoobject(magadan_response)

print(yakutsk_url)
print_json(yakutsk_response)
print(magadan_url)
print_json(magadan_response)
print("Якутск" if yandex_coords(yakutsk)[0] > yandex_coords(magadan)[0] else "Магадан")
print()

toronto_query = "Торонто"
home_city_url = build_url(YANDEX_GEOCODER_URL, {"apikey": YANDEX_GEOCODER_API_KEY, "geocode": browser_home_city, "format": "json", "lang": "ru_RU"})
toronto_url = build_url(YANDEX_GEOCODER_URL, {"apikey": YANDEX_GEOCODER_API_KEY, "geocode": toronto_query, "format": "json", "lang": "ru_RU"})
home_city_response = yandex_geocode(browser_home_city)
toronto_response = yandex_geocode(toronto_query)
home_city = yandex_geoobject(home_city_response)
toronto = yandex_geoobject(toronto_response)

print(home_city_url)
print_json(home_city_response)
print(toronto_url)
print_json(toronto_response)
print(browser_home_city if yandex_coords(home_city)[0] < yandex_coords(toronto)[0] else toronto_query)
print()

for city in ["Хабаровск", "Уфа", "Нижний Новгород", "Калининград", browser_home_city]:
    url = build_url(YANDEX_GEOCODER_URL, {"apikey": YANDEX_GEOCODER_API_KEY, "geocode": city, "format": "json", "lang": "ru_RU"})
    response = yandex_geocode(city)
    geoobject = yandex_geoobject(response)
    print(url)
    print_json(response)
    print(yandex_federal_district(geoobject))
print()

kemsu_query = "Кемерово, Красная улица, 6"
kemsu_url = build_url(YANDEX_GEOCODER_URL, {"apikey": YANDEX_GEOCODER_API_KEY, "geocode": kemsu_query, "format": "json", "lang": "ru_RU"})
kemsu_response = yandex_geocode(kemsu_query)
kemsu = yandex_geoobject(kemsu_response)
print(kemsu_url)
print_json(kemsu_response)
print(yandex_postcode(kemsu))


https://geocode-maps.yandex.ru/v1/?apikey=0aa0ce3c-c817-4d77-9cff-38f4b5a1bb38&geocode=%D0%AF%D0%BA%D1%83%D1%82%D1%81%D0%BA&format=json&lang=ru_RU
{
  "response": {
    "GeoObjectCollection": {
      "metaDataProperty": {
        "GeocoderResponseMetaData": {
          "request": "Якутск",
          "results": "10",
          "found": "10"
        }
      },
      "featureMember": [
        {
          "GeoObject": {
            "metaDataProperty": {
              "GeocoderMetaData": {
                "precision": "other",
                "text": "Россия, Республика Саха (Якутия), Якутск",
                "kind": "locality",
                "Address": {
                  "country_code": "RU",
                  "formatted": "Россия, Республика Саха (Якутия), Якутск",
                  "Components": [
                    {
                      "kind": "country",
                      "name": "Россия"
                    },
                    {
                      "kind": "province",


## Работа с API в Python

### №3
Напишите программу, которая на экране распечатает полный адрес и координаты Исторического музея города Москвы (Красная пл-дь, 1).


In [4]:
historical_museum = nominatim_search("Исторический музей Москва Красная площадь 1", limit=1)[0]
print("Полный адрес:", historical_museum["display_name"])
print("Координаты:", historical_museum["lat"], historical_museum["lon"])


Полный адрес: Исторический музей, 1, Красная площадь, Китай-город, 4, Тверской район, Москва, Центральный федеральный округ, 109012, Россия
Координаты: 55.7553230 37.6178815


### №4
Напишите программу, которая распечатает на экране к каким областям относятся города: Барнаул, Мелеуз, Йошкар-Ола.


In [5]:
for city in ["Барнаул", "Мелеуз", "Йошкар-Ола"]:
    record = nominatim_search(city, limit=1)[0]
    print(f"{city}: {region_name(record)}")


Барнаул: Алтайский край
Мелеуз: Башкортостан
Йошкар-Ола: Марий Эл


### №5
Напишите программу, которая распечатает на экране почтовый индекс Московского Уголовного Розыска (МУРа) «Петровки, 38».


In [6]:
mur = nominatim_search("Москва, Петровка, 38", limit=1)[0]
print(address_part(mur, "postcode"))


127051


### №6
Напишите программу, которая загрузит и сохранит в файл спутниковый снимок Австралии целиком.


In [7]:
australia_params = {
    "ll": "134,-25",
    "spn": "45,28",
    "l": "sat",
    "size": "650,450",
}
download_file(STATIC_MAP_URL, "australia_satellite.png", australia_params)


PosixPath('/Users/ivan/Documents/Github/Python/australia_satellite.png')

### №7
Напишите программу, которая загрузит и сохранит в файл карту города Кемерово со следующими отметками (как ставить отметки посмотрите в документации):

a) ЖД Вокзал
b) Кемеровский кардиологический диспансер
c) Музей-заповедник «Красная Горка»
d) Какой-нибудь парк на ваш выбор


In [8]:
kemerovo_points = {
    "ЖД вокзал": (86.0746, 55.3337),
    "Кемеровский кардиологический диспансер": (86.1241, 55.3458),
    "Музей-заповедник Красная Горка": (86.0824, 55.3613),
    "Парк Чудес": (86.0768, 55.3544),
}

pt = "~".join(
    f"{lon},{lat},pm2rdm"
    for lon, lat in kemerovo_points.values()
)

map_path = download_file(
    STATIC_MAP_URL,
    "kemerovo_points.png",
    {
        "ll": "86.0872,55.3547",
        "spn": "0.35,0.22",
        "l": "map",
        "pt": pt,
        "size": "650,450",
    },
)

print(map_path)
for title, (lon, lat) in kemerovo_points.items():
    print(f"{title}: {lat}, {lon}")


/Users/ivan/Documents/Github/Python/kemerovo_points.png
ЖД вокзал: 55.3337, 86.0746
Кемеровский кардиологический диспансер: 55.3458, 86.1241
Музей-заповедник Красная Горка: 55.3613, 86.0824
Парк Чудес: 55.3544, 86.0768


### №8
Напишите программу, которая загрузит и сохранит в файл карту Кемеровской области целиком, с нанесенной на нее ломанной линией маршрута: Кемерово - Ленинск-Кузнецк - Новокузнецк - Шерегеш.


In [9]:
route_kuzbass = [
    (86.0872, 55.3547),  # Кемерово
    (86.1622, 54.6636),  # Ленинск-Кузнецк
    (87.1099, 53.7576),  # Новокузнецк
    (87.9869, 52.9209),  # Шерегеш
]

pl = "c:ff0000AA,w:5," + ",".join(
    f"{lon},{lat}"
    for lon, lat in route_kuzbass
)

map_path = download_file(
    STATIC_MAP_URL,
    "kuzbass_route.png",
    {
        "ll": "87.0,54.2",
        "spn": "3.4,3.1",
        "l": "map",
        "pl": pl,
        "size": "650,450",
    },
)
print(map_path)


/Users/ivan/Documents/Github/Python/kuzbass_route.png


### №9
Напишите программу, которая определяет, какой из списка городов расположен южнее всех остальных. Список городов вводится через запятую.


In [10]:
def southernmost_city(cities_text):
    cities = [city.strip() for city in cities_text.split(",") if city.strip()]
    if not cities:
        raise ValueError("Введите хотя бы один город")

    city_coordinates = []
    for city in cities:
        record = nominatim_search(city, limit=1)[0]
        lat, lon = record_coords(record)
        city_coordinates.append((city, lat, lon))

    return min(city_coordinates, key=lambda item: item[1])


cities_text = "Кемерово, Новосибирск, Москва, Сочи, Владивосток"
city, lat, lon = southernmost_city(cities_text)
print(f"Самый южный город: {city} ({lat}, {lon})")


Самый южный город: Владивосток (43.1150678, 131.8855768)


### №10
Определите длину пути, заданного последовательностью точек. Сохраните в файл карту с ломанной линией заданного пути, в его средней точке должна стоять метка.


In [11]:
from math import asin, cos, radians, sin, sqrt


def haversine_km(point_a, point_b):
    lon1, lat1 = point_a
    lon2, lat2 = point_b
    radius = 6371.0
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * radius * asin(sqrt(a))


def path_length_km(points):
    return sum(haversine_km(a, b) for a, b in zip(points, points[1:]))


def middle_point(points):
    total = path_length_km(points)
    half = total / 2
    passed = 0

    for start, end in zip(points, points[1:]):
        segment = haversine_km(start, end)
        if passed + segment >= half:
            ratio = (half - passed) / segment
            lon = start[0] + (end[0] - start[0]) * ratio
            lat = start[1] + (end[1] - start[1]) * ratio
            return lon, lat
        passed += segment

    return points[-1]


path_points = [
    (86.0872, 55.3547),
    (86.1622, 54.6636),
    (87.1099, 53.7576),
    (87.9869, 52.9209),
]

length = path_length_km(path_points)
mid_lon, mid_lat = middle_point(path_points)
pl = "c:0066ffAA,w:5," + ",".join(f"{lon},{lat}" for lon, lat in path_points)
pt = f"{mid_lon},{mid_lat},pm2rdm"

map_path = download_file(
    STATIC_MAP_URL,
    "path_with_middle_point.png",
    {
        "ll": "87.0,54.2",
        "spn": "3.4,3.1",
        "l": "map",
        "pl": pl,
        "pt": pt,
        "size": "650,450",
    },
)

print(f"Длина пути: {length:.2f} км")
print(f"Средняя точка: {mid_lat:.6f}, {mid_lon:.6f}")
print(map_path)


Длина пути: 304.84 км
Средняя точка: 54.084947, 86.767487
/Users/ivan/Documents/Github/Python/path_with_middle_point.png


## Web Scraping в Python

### №11
Напишите программу, которая выведет на экран все ссылки со страницы `http://olympus.realpython.org/profiles`, ориентируясь на атрибут `href` у HTML-тега `a`.

Вывод должен быть следующим:

`http://olympus.realpython.org/profiles/aphrodite`
`http://olympus.realpython.org/profiles/poseidon`
`http://olympus.realpython.org/profiles/dionysus`


In [12]:
olympus_url = "http://olympus.realpython.org/profiles"
olympus_html = fetch_text(olympus_url)
links = re.findall(r'<a[^>]+href=["\']([^"\']+)["\']', olympus_html)

for link in links:
    print(urljoin(olympus_url, link))


http://olympus.realpython.org/profiles/aphrodite
http://olympus.realpython.org/profiles/poseidon
http://olympus.realpython.org/profiles/dionysus


### №12
Напишите программу, которая вытащит список всех авторов цитат с сайта `https://quotes.toscrape.com/` и выведет его на экран, отсортированным по уменьшению количества цитат автора, т.е. самым первым должен быть автор с наибольшим числом цитат на сайте.

Обратите внимание, что это многостраничный сайт.


In [13]:
def collect_authors():
    base_url = "https://quotes.toscrape.com/"
    page_url = base_url
    counter = Counter()

    while page_url:
        html = fetch_text(page_url)
        authors = re.findall(
            r'<small class="author" itemprop="author">([^<]+)</small>',
            html,
        )
        counter.update(unescape(author) for author in authors)

        next_page = re.search(r'<li class="next"><a href="([^"]+)"', html)
        page_url = urljoin(base_url, next_page.group(1)) if next_page else None

    return counter


author_counter = collect_authors()
for author, _ in sorted(author_counter.items(), key=lambda item: (-item[1], item[0])):
    print(author)


Albert Einstein
André Gide
Eleanor Roosevelt
J.K. Rowling
Jane Austen
Marilyn Monroe
Steve Martin
Thomas A. Edison
